# CS-4063 NLP — Assignment 2 notebook

Run cells top to bottom. Training cells call the same scripts as the CLI (`a.py` … `j.py`). If checkpoints already exist, you may skip heavy cells and only run the metrics and comparison cells.

In [ ]:
import subprocess
import sys


def run_script(name):
    print("===", name, "===", flush=True)
    r = subprocess.run([sys.executable, name + ".py"], cwd=".")
    if r.returncode != 0:
        raise RuntimeError("failed " + name)


for s in ["a", "b", "c", "d"]:
    run_script(s)

In [ ]:
run_script("e")
run_script("f")

In [ ]:
run_script("g")

In [ ]:
run_script("i")
import importlib.util

spec = importlib.util.spec_from_file_location("h", "h.py")
h = importlib.util.module_from_spec(spec)
spec.loader.exec_module(h)
print("h module loaded", h.TransformerCls)
run_script("j")

In [ ]:
import json
import os
import torch

from IPython.display import Markdown, display

M = {}
if os.path.isfile("models/metrics_summary.json"):
    M = json.load(open("models/metrics_summary.json", encoding="utf-8"))
T = {}
if os.path.isfile("models/transformer_cls.pt"):
    try:
        T = torch.load("models/transformer_cls.pt", map_location="cpu", weights_only=False)
    except TypeError:
        T = torch.load("models/transformer_cls.pt", map_location="cpu")
print("metrics_summary keys", list(M.keys()))
print("transformer_ckpt keys", [k for k in T.keys() if k != "state_dict"])

In [ ]:
pta = float(M.get("pos_test_token_acc", float("nan")))
pmf = float(M.get("pos_test_macro_f1", float("nan")))
nf = float(M.get("ner_overall_micro_f1_crf", float("nan")))
ta = float(T.get("test_acc", float("nan")))
tf1 = float(T.get("test_macro_f1", float("nan")))

text = """
## BiLSTM vs Transformer (answers)

1. **Accuracy:** The topic classifier is trained only on the Transformer; its test accuracy is **{ta:.4f}** with macro-F1 **{tf1:.4f}**. The BiLSTM checkpoints target different tasks: POS token accuracy is **{pta:.4f}** with macro-F1 **{pmf:.4f}**, and NER (CRF decode) overall entity micro-F1 is **{nf:.4f}**, so headline “accuracy” is not directly comparable across task boundaries, but the Transformer’s number answers the document-label task while the BiLSTM numbers summarize token labeling quality.

2. **Epochs to converge:** The Transformer uses a fixed **20**-epoch schedule with cosine learning rate and warmup; the BiLSTMs in `f.py` use early stopping on validation macro-F1 with patience, so they often finish sooner than the full 60-epoch cap when validation flattens. In practice the Transformer therefore runs the full budget unless you shorten it, whereas the BiLSTM runs until early stopping triggers.

3. **Time per epoch:** A single Transformer epoch scans the small article tensor dataset with self-attention over length 256, matrix multiplies dominate, and there is no recurrent unroll; the BiLSTM batches variable-length sentences with packed LSTM steps plus (for NER) CRF likelihoods. Per-epoch wall time is usually lower for the BiLSTM on this corpus size because its sequence lengths are shorter on average and the model is narrower, while the Transformer pays a quadratic attention cost in the sequence length even when many positions are padding-neutralized.

4. **Attention heatmaps:** The saved `models/transformer_attn_article_*.png` plots use the CLS row of attention from two heads in the last encoder block. They typically emphasize a handful of content tokens (often topical nouns or entities) and down-weight padding; differences between heads show complementary focus patterns rather than one uniform bag of keywords.

5. **Small data (200–300 articles):** With only a few hundred labeled documents, a smaller BiLSTM tagger with pretrained Word2Vec can be easier to regularize and cheaper to iterate on for POS/NER, while a Transformer encoder offers more capacity and global mixing but overfits faster without strong regularization and data. For **topic classification** specifically, the Transformer is still viable if weight decay, dropout, and early monitoring on validation are used; for **low-resource tagging**, the BiLSTM plus frozen embeddings is often the more robust default.
""".format(
    ta=ta,
    tf1=tf1,
    pta=pta,
    pmf=pmf,
    nf=nf,
)

display(Markdown(text))